In [ ]:
import traceback, json as _json
_result = {"status": "ok", "log": []}
def _log(*a):
    _result["log"].append(" ".join(str(x) for x in a))
def _save():
    import notebookutils
    notebookutils.fs.put("Files/ci_test_result.json", _json.dumps(_result, indent=1), True)

NEW_INSTRUCTIONS = """You are RetailEvidence, a Fabric data agent for a synthetic UK retail demonstration.
Use only the attached Fabric Lakehouse tables. Revenue is SUM(fact_daily_sales.Revenue), not SUM(Units).
For year-over-year September electronics comparison, filter dim_product.Category = Electronics, compare September 2025 vs September 2026.
Join sales to dim_product on ProductId, and to dim_store on StoreId.
Inventory is a single dated snapshot; never infer month-long shortages from it.
Promotions and store operations are observational data; correlation does not establish causality.
When asked about newly released audit observations, query new_evidence and state how many rows were found.
Always show periods, denominators, relevant store IDs, source table names, and caveats.
Never invent figures or claim a business action was executed. If data is missing, say so.

CODE INTERPRETER
Use Code Interpreter, when available, for analytical processing that improves the investigation:
- Year-over-year percentage changes and deltas.
- Ranking stores by absolute and percentage revenue decline.
- Group statistics (low-stock stores vs fully stocked stores).
- Identifying anomalies and outliers in query results.
- Creating comparison charts from retrieved data, where supported (label axes, periods, units).
Inputs to Code Interpreter must be actual query results from the attached tables.
Never generate fictional records or replace missing data with assumptions.
After computing, verify totals match the queried values; round percentages to one decimal place.
When you use Code Interpreter, state which figures came from Fabric data and which calculations it performed."""

try:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-q",
                    "fabric-data-agent-sdk"], check=True)

    from fabric.dataagent.client import FabricDataAgentManagement, FabricOpenAIResponses
    agent = FabricDataAgentManagement("RetailEvidence")
    agent.update_configuration(instructions=NEW_INSTRUCTIONS)
    _log("instructions updated, len:", len(NEW_INSTRUCTIONS))

    # log current config keys to spot any code-interpreter toggle the SDK exposes
    cfg = agent.get_configuration()
    try:
        _result["agent_config"] = {k: str(v)[:200] for k, v in dict(cfg).items()}
    except Exception:
        _result["agent_config"] = str(cfg)[:1000]

    agent.publish()
    _log("republished")
    _save()

    client = FabricOpenAIResponses(artifact_name="RetailEvidence")
    conv = client.conversations.create()
    q = ("Retrieve September 2025 and September 2026 electronics revenue by store. "
         "Use Code Interpreter, if available, to calculate year-over-year percentage "
         "changes, rank stores by revenue decline, and create a comparison chart. "
         "Explain which results come from Fabric data and which calculations were "
         "performed by Code Interpreter.")
    resp = client.responses.create(input=q, conversation=conv.id)
    _result["answer"] = resp.output_text
    _log("answered, chars:", len(resp.output_text))
    _save()

    # inspect diagnostics for actual tool executions
    try:
        payload = client.diagnostics.get(conv.id, response_id=resp.id)
        raw = _json.dumps(payload, default=str)
        _result["diag_size"] = len(raw)
        low = raw.lower()
        _result["mentions_code_interpreter"] = ("code_interpreter" in low or
                                                "codeinterpreter" in low or
                                                "python" in low)
        # collect tool-ish keys for visibility
        import re
        tools = sorted(set(re.findall(r'"(?:tool[_a-z]*|name|type)"\s*:\s*"([^"]{1,60})"', raw)))
        _result["tool_markers"] = tools[:60]
        _log("diagnostics captured")
    except Exception:
        _log("diagnostics failed:", traceback.format_exc()[-400:])
except Exception:
    _result["status"] = "error"
    _result["log"].append(traceback.format_exc())

_save()
print(_result["status"])
